# MERSCOPE resegmentation tuning on a small ROI

This notebook is for **interactive parameter tuning** before running resegmentation on the full dataset.

It lets you:
1. Load and inspect dataset-level transcript/image overview.
2. Pick a small ROI by global coordinates (`x/y` in microns).
3. Run each step of the MERSCOPE reseg pipeline on that ROI:
   - z-range max projection
   - Cellpose
   - mask filtering / polygon creation
   - transcript-to-cell assignment
   - optional ProSeg run on the ROI

The notebook does **not** modify the original raw dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import dask
import dask.array as da
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from shapely.geometry import Polygon
from skimage.measure import find_contours, label, regionprops
from skimage.transform import resize
from skimage.segmentation import find_boundaries
from PIL import Image

import spatialdata as sd
from cellpose import io, models

from pathlib import Path
import sys

# robustly find repo root
repo = Path.cwd()
if not (repo / "src" / "resegmentation" / "proseg_wrapper.py").exists():
    repo = repo.parent

sys.path.insert(0, str(repo / "src" / "resegmentation"))
from proseg_wrapper import run_proseg_refinement

sns.set_context('notebook')
plt.rcParams['figure.figsize'] = (8, 5)

# Recommended for some spatialdata+dask stacks.
dask.config.set({'dataframe.query-planning': False})

In [ ]:
# Load SpatialData and basic metadata.
sdata = sd.read_zarr(ZARR_PATH)
manifest = json.loads(MANIFEST_PATH.read_text())
microns_per_pixel = float(manifest['microns_per_pixel'])
M = np.loadtxt(TRANSFORM_PATH)  # micron_to_mosaic_pixel_transform

print('Images:', list(sdata.images.keys()))
print('Points:', list(sdata.points.keys()))
print('Shapes:', list(sdata.shapes.keys()))
print('Tables:', list(sdata.tables.keys()))
print('microns_per_pixel:', microns_per_pixel)
print('transform matrix:\n', M)

In [ ]:
def list_plane_keys(images, prefix=None):
    pat = re.compile(r'^(?P<prefix>.+)_z(?P<z>\d+)$')
    out = []
    for k in images.keys():
        m = pat.match(str(k))
        if not m:
            continue
        if prefix is not None and not str(k).startswith(prefix):
            continue
        out.append((int(m.group('z')), str(k)))
    return sorted(out)

def extract_image_array(img_obj):
    img_xr = img_obj
    if hasattr(img_obj, '__contains__') and 'scale0' in img_obj:
        img_xr = img_obj['scale0'].ds['image']
    else:
        try:
            img_xr = img_obj['scale0'].ds['image']
        except Exception:
            pass

    if hasattr(img_xr, 'dims') and all(d in img_xr.dims for d in ('y', 'x', 'c')):
        arr = img_xr.transpose('y', 'x', 'c').data.compute()
        c_coords = [str(x) for x in img_xr.coords['c'].values] if 'c' in img_xr.coords else None
        return arr, c_coords
    if hasattr(img_xr, 'dims') and all(d in img_xr.dims for d in ('c', 'y', 'x')):
        arr = np.moveaxis(img_xr.data.compute(), 0, -1)
        c_coords = [str(x) for x in img_xr.coords['c'].values] if 'c' in img_xr.coords else None
        return arr, c_coords

    arr = img_xr.data.compute() if hasattr(img_xr, 'data') else np.asarray(img_xr)
    if arr.ndim == 2:
        arr = arr[..., np.newaxis]
    elif arr.ndim == 3 and arr.shape[0] <= 8 and arr.shape[-1] > 8:
        arr = np.moveaxis(arr, 0, -1)
    c_coords = [f'c{i}' for i in range(arr.shape[-1])]
    return arr, c_coords

def global_to_pixel(x_global, y_global, M):
    arr = np.vstack([x_global, y_global, np.ones_like(x_global)])
    out = M @ arr
    return out[0], out[1]

def pixel_to_global(x_pix, y_pix, Minv):
    arr = np.vstack([x_pix, y_pix, np.ones_like(x_pix)])
    out = Minv @ arr
    return out[0], out[1]

def filter_masks_basic(
    seg_masks,
    max_eccentricity=0.95,
    n_jobs=None,
    show_progress=False,
    min_area_percentile=10.0,
    min_area_px=None,
):
    """Use repository implementation for mask filtering.

    Accepts notebook args for compatibility and tuning.
    """
    from reseg_tools import filter_cell_by_regionprops as _repo_filter_masks

    return _repo_filter_masks(
        seg_masks,
        max_eccentricity=max_eccentricity,
        n_jobs=n_jobs,
        show_progress=bool(show_progress),
        min_area_percentile=min_area_percentile,
        min_area_px=min_area_px,
    )

def masks_to_polygons(seg_masks, factor_rescale=0, scale_factor=None, show_progress=None, n_jobs=None):
    """Use repository implementation for mask->polygon conversion.

    Accepts legacy notebook args (`scale_factor`, `show_progress`) for compatibility.
    """
    from reseg_tools import masks_to_polygons as _repo_masks_to_polygons

    if scale_factor is not None:
        factor_rescale = scale_factor

    return _repo_masks_to_polygons(
        seg_masks,
        factor_rescale=factor_rescale,
        n_jobs=n_jobs,
        show_progress=bool(show_progress),
    )

In [ ]:
# ---- Dataset-level overview for ROI selection ----
points_key = list(sdata.points.keys())[0]
pts_dd = sdata.points[points_key]
sample_frac = 0.005  # increase for denser overview
overview = pts_dd.sample(frac=sample_frac, random_state=42)[['x', 'y', 'z', 'gene']].compute()

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(overview['x'], overview['y'], s=0.2, alpha=0.3)
ax.set_title(f'Global transcript overview (sample frac={sample_frac})')
ax.set_xlabel('global x (microns)')
ax.set_ylabel('global y (microns)')
ax.set_aspect('equal')
plt.show()

In [ ]:
# ---- Set your ROI in GLOBAL micron coordinates ----
# Edit these values iteratively and rerun downstream cells.
ROI = {
    'x_min': 5000.0,
    'x_max': 6200.0,
    'y_min': 3500.0,
    'y_max': 4700.0,
}

z_start = 0
z_end = 6
channel_names_to_use = ['DAPI', 'PolyT']  # subset from available channels

print(ROI)
print('z range:', z_start, z_end)
print('channels:', channel_names_to_use)

In [ ]:
# Visualize selected ROI over transcript overview.
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(overview['x'], overview['y'], s=0.2, alpha=0.25)
rect_x = [ROI['x_min'], ROI['x_max'], ROI['x_max'], ROI['x_min'], ROI['x_min']]
rect_y = [ROI['y_min'], ROI['y_min'], ROI['y_max'], ROI['y_max'], ROI['y_min']]
ax.plot(rect_x, rect_y, color='red', linewidth=2)
ax.set_title('ROI on global transcript overview')
ax.set_xlabel('global x (microns)')
ax.set_ylabel('global y (microns)')
ax.set_aspect('equal')
plt.show()

In [ ]:
import dask.array as da

# ---- Build z-range max projection and crop to ROI (lazy ROI reads) ----
plane_keys = list_plane_keys(sdata.images, prefix='202509261108_P7513-2_VMSC19502_region_R2')
selected_keys = [k for z, k in plane_keys if z_start <= z <= z_end]
print('selected plane keys:', selected_keys)

if not selected_keys:
    raise ValueError('No image planes selected for the z range.')

# Convert ROI global microns -> mosaic pixel bbox first.
xpix, ypix = global_to_pixel(
    np.array([ROI['x_min'], ROI['x_max']]),
    np.array([ROI['y_min'], ROI['y_max']]),
    M,
)
x0, x1 = int(np.floor(min(xpix))), int(np.ceil(max(xpix)))
y0, y1 = int(np.floor(min(ypix))), int(np.ceil(max(ypix)))

# Clamp using first selected plane shape without loading pixel data.
img0 = sdata.images[selected_keys[0]]
img0_xr = img0['scale0'].ds['image'] if hasattr(img0, '__contains__') and 'scale0' in img0 else img0
if all(d in img0_xr.dims for d in ('y', 'x')):
    h = int(img0_xr.sizes['y'])
    w = int(img0_xr.sizes['x'])
else:
    raise ValueError(f'Unexpected image dims for ROI crop: {img0_xr.dims}')

x0 = max(0, x0)
y0 = max(0, y0)
x1 = min(w, x1)
y1 = min(h, y1)

print('pixel bbox:', (x0, x1, y0, y1), 'size=', (x1 - x0, y1 - y0))

if x1 <= x0 or y1 <= y0:
    raise ValueError('ROI maps to an empty pixel crop. Adjust ROI or transform.')

lazy_roi_planes = []
channels = None
use_ch = None

for key in selected_keys:
    img_obj = sdata.images[key]
    img_xr = img_obj['scale0'].ds['image'] if hasattr(img_obj, '__contains__') and 'scale0' in img_obj else img_obj

    if not all(d in img_xr.dims for d in ('y', 'x', 'c')):
        raise ValueError(f'Expected image dims to include c,y,x. Got {img_xr.dims} for {key}')

    c_all = [str(c) for c in img_xr.coords['c'].values] if 'c' in img_xr.coords else None
    if channels is None:
        channels = c_all

    roi_xr = img_xr.isel(y=slice(y0, y1), x=slice(x0, x1))

    if channel_names_to_use is not None and c_all is not None:
        keep = [c for c in channel_names_to_use if c in c_all]
        if not keep:
            raise ValueError('No selected channels found in image channel coords.')
        roi_xr = roi_xr.sel(c=keep)
        if use_ch is None:
            use_ch = keep
    else:
        if use_ch is None:
            use_ch = c_all

    # Keep this lazy; materialize only after z-stack max projection.
    lazy_roi_planes.append(roi_xr.transpose('y', 'x', 'c').data)

proj = da.max(da.stack(lazy_roi_planes, axis=0), axis=0).compute()  # (y, x, c)
crop = proj

print('projection/crop shape:', crop.shape)
print('available channels:', channels)
print('channels used:', use_ch)

Minv = np.linalg.inv(M)

fig, ax = plt.subplots(figsize=(7, 7))
show = crop[..., 0] if crop.shape[-1] > 0 else crop.squeeze()
ax.imshow(show, cmap='gray')
ax.set_title('ROI crop preview (first channel)')
ax.axis('off')
plt.show()

In [ ]:
recompute_cellpose = True
if recompute_cellpose:
    # ---- Cellpose tuning cell ----
    cellpose_params = {
        'model_type': 'cyto3',
        'gpu': True,
        'diameter': None,
        'flow_threshold': 0.8,
        'cellprob_threshold': -5.0,
        'tile_overlap': 0.15,
        'bsize': 256,
        'factor_rescale': 1.0,  # set >1 to downscale before segmentation
    }

    # Preprocess crop into 3-channel uint8 image for Cellpose.
    img = crop.astype(np.float32)
    p2, p98 = np.percentile(img, (2, 98))
    img = np.clip((img - p2) / (p98 - p2 + 1e-8), 0, 1)
    img8 = (img * 255).astype(np.uint8)
    if img8.ndim == 2:
        img8 = np.stack([img8] * 3, axis=-1)
    elif img8.shape[-1] == 1:
        img8 = np.repeat(img8, 3, axis=-1)
    elif img8.shape[-1] == 2:
        img8 = np.concatenate([img8, np.zeros_like(img8[..., :1])], axis=-1)
    elif img8.shape[-1] > 3:
        img8 = img8[..., :3]

    scale_factor = float(cellpose_params['factor_rescale'])
    if scale_factor > 1.0:
        target_shape = (int(img8.shape[0] / scale_factor), int(img8.shape[1] / scale_factor), img8.shape[2])
        img_seg = resize(img8, target_shape, preserve_range=True, anti_aliasing=True).astype(np.uint8)
    else:
        img_seg = img8

    model = models.CellposeModel(model_type=cellpose_params['model_type'], gpu=cellpose_params['gpu'])
    masks, flows, styles = model.eval(
        img_seg,
        diameter=cellpose_params['diameter'],
        flow_threshold=cellpose_params['flow_threshold'],
        cellprob_threshold=cellpose_params['cellprob_threshold'],
        tile_overlap=cellpose_params['tile_overlap'],
        bsize=cellpose_params['bsize'],
    )

    print('raw mask labels:', int(masks.max()))
    bound = find_boundaries(masks > 0, mode='outer')
    overlay = img_seg[..., 0].copy()
    overlay = np.stack([overlay, overlay, overlay], axis=-1)
    overlay[bound] = [255, 0, 0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img_seg[..., 0], cmap='gray')
    axes[0].set_title('Cellpose input')
    axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title('Cellpose boundaries overlay')
    axes[1].axis('off')
    plt.tight_layout()

    labeled = label(masks)
    regions = regionprops(labeled)
    # create histogram of eccentricity of each region
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.hist([region.eccentricity for region in regions], bins=20)
    ax.set_title('Eccentricity histogram')
    plt.show()

In [ ]:
recompute_polygons = True
if recompute_polygons:
    # ---- Filter masks and build polygons ----
    max_eccentricity = 0.99 # tune this
    min_area_percentile = 1.0  # lower = keep more small cells
    min_area_px = None          # set e.g. 80 to force absolute threshold
    filtered_masks = filter_masks_basic(
        masks,
        max_eccentricity=max_eccentricity,
        show_progress=True,
        n_jobs=16,
        min_area_percentile=min_area_percentile,
        min_area_px=min_area_px,
    )
    print('filtered mask labels:', int(filtered_masks.max()))
    # If image was downscaled before Cellpose, scale polygons back to ROI-pixel coordinates.
    poly_scale = scale_factor if scale_factor > 1.0 else 1.0
    polygons_local = masks_to_polygons(filtered_masks, scale_factor=poly_scale, show_progress=True, n_jobs=16)
    cells_local = gpd.GeoDataFrame({
    'cell_id': [f'cell_{i+1}' for i in range(len(polygons_local))],
    'geometry': polygons_local,
    })
    print('polygon count:', len(cells_local))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img8[..., 0], cmap='gray')
if len(cells_local) > 0:
    cells_local.boundary.plot(ax=ax, linewidth=0.5)
    ax.set_title('Filtered polygons on ROI')
    ax.axis('off')
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img8[..., 0], cmap='gray')
if len(cells_local) > 0:
    cells_local.boundary.plot(ax=ax, linewidth=2)
    ax.set_title('Filtered polygons on ROI')
    ax.axis('off')
    plt.xlim(0, 1000)
    plt.ylim(0, 1000)
    plt.show()

In [ ]:
# ---- Checkpoint: save or reload post-polygon outputs ----
# Put this cell right after "Filter masks and build polygons"
from pathlib import Path
import numpy as np
import geopandas as gpd
import pickle

CHECKPOINT_DIR = Path("./_roi_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Use a name that encodes the ROI/z-range so you can keep multiple checkpoints
ckpt_name = f"roi_x{ROI['x_min']:.0f}_{ROI['x_max']:.0f}_y{ROI['y_min']:.0f}_{ROI['y_max']:.0f}_z{z_start}_{z_end}"
base = CHECKPOINT_DIR / ckpt_name

SAVE_CHECKPOINT = True   # set True after long cell finishes
LOAD_CHECKPOINT = False  # set True on notebook restart to skip recomputing

if SAVE_CHECKPOINT:
    # arrays
    np.savez_compressed(
        f"{base}_arrays.npz",
        filtered_masks=filtered_masks.astype(np.int32),
        masks=masks.astype(np.int32),
    )

    # polygons/cells
    cells_local.to_parquet(f"{base}_cells_local.parquet", index=False)

    # optional extras that help keep context consistent
    meta = {
        "x0": int(x0), "x1": int(x1), "y0": int(y0), "y1": int(y1),
        "poly_scale": float(poly_scale),
        "z_start": int(z_start), "z_end": int(z_end),
        "points_key": points_key,
        "channel_names_to_use": channel_names_to_use,
        "ROI": dict(ROI),
    }
    with open(f"{base}_meta.pkl", "wb") as f:
        pickle.dump(meta, f)

    print("Saved checkpoint:")
    print(f"  {base}_arrays.npz")
    print(f"  {base}_cells_local.parquet")
    print(f"  {base}_meta.pkl")

if LOAD_CHECKPOINT:
    arr = np.load(f"{base}_arrays.npz")
    filtered_masks = arr["filtered_masks"]
    masks = arr["masks"]

    cells_local = gpd.read_parquet(f"{base}_cells_local.parquet")
    # geometry column usually restores automatically; if not:
    if "geometry" not in cells_local.columns:
        raise ValueError("Checkpoint load failed: geometry column missing in cells_local.")

    polygons_local = list(cells_local.geometry)

    with open(f"{base}_meta.pkl", "rb") as f:
        meta = pickle.load(f)

    # restore context vars used by downstream cells
    x0, x1, y0, y1 = meta["x0"], meta["x1"], meta["y0"], meta["y1"]
    poly_scale = meta["poly_scale"]
    z_start, z_end = meta["z_start"], meta["z_end"]
    points_key = meta["points_key"]
    channel_names_to_use = meta["channel_names_to_use"]
    ROI = meta["ROI"]

    print("Loaded checkpoint:", base)
    print("filtered_masks shape:", filtered_masks.shape)
    print("polygon count:", len(cells_local))

In [ ]:
# ---- Transcript assignment for ROI ----
# Filter transcripts by global micron ROI, map to local ROI pixels, then spatially join.
pts_full = sdata.points[points_key][['x', 'y', 'z', 'gene']].compute()
pts_roi = pts_full[
    (pts_full['x'] >= ROI['x_min']) & (pts_full['x'] <= ROI['x_max']) &
    (pts_full['y'] >= ROI['y_min']) & (pts_full['y'] <= ROI['y_max'])
].copy()

xpix_roi, ypix_roi = global_to_pixel(pts_roi['x'].to_numpy(), pts_roi['y'].to_numpy(), M)
pts_roi['x_local_px'] = xpix_roi - x0
pts_roi['y_local_px'] = ypix_roi - y0

pts_gdf = gpd.GeoDataFrame(
    pts_roi,
    geometry=gpd.points_from_xy(pts_roi['x_local_px'], pts_roi['y_local_px'])
)

if len(cells_local) > 0:
    # Use intersects so boundary transcripts are not dropped from seeds.
    join = gpd.sjoin(pts_gdf, cells_local[['cell_id', 'geometry']], how='left', predicate='intersects')
    # sjoin can return >1 row per point if polygons overlap; map back by point index.
    point_to_cell = join.groupby(join.index)['cell_id'].first()
    pts_gdf['cell_id'] = point_to_cell.reindex(pts_gdf.index)
else:
    pts_gdf['cell_id'] = np.nan

assigned = pts_gdf['cell_id'].notna().sum()
print('ROI transcripts:', len(pts_gdf))
print('Assigned transcripts:', int(assigned), f'({assigned / max(len(pts_gdf), 1):.2%})')

# Build ROI-level count matrix
roi_counts = pd.crosstab(pts_gdf['cell_id'], pts_gdf['gene'])
roi_counts = roi_counts.loc[[idx for idx in roi_counts.index if pd.notna(idx)]]
print('ROI count matrix shape:', roi_counts.shape)
display(roi_counts.head())

In [ ]:
# ---- Optional: run ProSeg on ROI transcripts ----
# Edit run_proseg=False to skip this cell safely.
run_proseg = True
proseg_binary = '/home/becalia/.cargo/bin/proseg'  # edit if needed
proseg_output_dir = Path('./tmp_proseg_roi_output_params')
write_latest_zarr = False  # rewrite to latest SpatialData zarr format after ProSeg

# Use all z planes present in the ROI transcripts.
if 'pts_gdf' in globals() and 'z' in pts_gdf.columns:
    z_vals = pd.to_numeric(pts_gdf['z'], errors='coerce').dropna().unique()
    voxel_layers_default = int(len(np.unique(z_vals))) if len(z_vals) > 0 else 1
else:
    voxel_layers_default = 1
print('ProSeg voxel_layers:', voxel_layers_default)

# Prepare Cellpose mask inputs for ProSeg.
cellpose_masks = None
cellpose_scale = None
cellpose_x_transform = None
cellpose_y_transform = None

if 'filtered_masks' in globals():
    cellpose_masks = filtered_masks
elif 'masks' in globals():
    cellpose_masks = masks

if cellpose_masks is not None:
    # Transcripts are in local ROI pixel coords (x_local_px, y_local_px).
    # Cellpose masks are also in local pixel coords (possibly downscaled by scale_factor).
    # The transform maps mask pixels → transcript coords, so it's just the rescale factor.
    sf = float(scale_factor) if 'scale_factor' in globals() and float(scale_factor) > 1.0 else 1.0
    Minv_local = np.linalg.inv(M)
    S = np.array([
        [sf, 0.0, float(x0)],
        [0.0, sf, float(y0)],
        [0.0, 0.0, 1.0],
    ], dtype=float)
    T = Minv_local @ S
    cellpose_x_transform = (float(T[0, 0]), float(T[0, 1]), float(T[0, 2]))
    cellpose_y_transform = (float(T[1, 0]), float(T[1, 1]), float(T[1, 2]))
    cellpose_scale = None

    print('Cellpose mask transform (affine to microns):')
    print('  x_transform:', cellpose_x_transform)
    print('  y_transform:', cellpose_y_transform)

# Define previous sweep parameter sets (kept for reference, not used).
proseg_param_sets_prev = [
    {
        'label': 'base',
        'samples': 500,
        'voxel_size': 2.0,
        'burnin_voxel_size': 4.0,
        'voxel_layers': voxel_layers_default,
        'nuclear_reassignment_prob': 0.25,
        'diffusion_probability': 0.25,
        'cell_compactness': None,
        'num_threads': 64,
    },
    {
        'label': 'smaller_voxels',
        'samples': 500,
        'voxel_size': 1.0,
        'burnin_voxel_size': 2.0,
        'voxel_layers': voxel_layers_default,
        'nuclear_reassignment_prob': 0.25,
        'diffusion_probability': 0.25,
        'cell_compactness': None,
        'num_threads': 64,
    },
    {
        'label': 'larger_voxels',
        'samples': 500,
        'voxel_size': 3.0,
        'burnin_voxel_size': 6.0,
        'voxel_layers': voxel_layers_default,
        'nuclear_reassignment_prob': 0.25,
        'diffusion_probability': 0.25,
        'cell_compactness': None,
        'num_threads': 64,
    },
    {
        'label': 'more_diffusion',
        'samples': 500,
        'voxel_size': 2.0,
        'burnin_voxel_size': 4.0,
        'voxel_layers': voxel_layers_default,
        'nuclear_reassignment_prob': 0.25,
        'diffusion_probability': 0.5,
        'cell_compactness': None,
        'num_threads': 64,
    },
    {
        'label': 'more_reassign',
        'samples': 500,
        'voxel_size': 2.0,
        'burnin_voxel_size': 4.0,
        'voxel_layers': voxel_layers_default,
        'nuclear_reassignment_prob': 0.5,
        'diffusion_probability': 0.25,
        'cell_compactness': None,
        'num_threads': 64,
    },
    {
        'label': 'compact_0.1',
        'samples': 500,
        'voxel_size': 2.0,
        'burnin_voxel_size': 4.0,
        'voxel_layers': voxel_layers_default,
        'nuclear_reassignment_prob': 0.25,
        'diffusion_probability': 0.25,
        'cell_compactness': 0.1,
        'num_threads': 64,
    },
    {
        'label': 'compact_0.3',
        'samples': 500,
        'voxel_size': 2.0,
        'burnin_voxel_size': 4.0,
        'voxel_layers': voxel_layers_default,
        'nuclear_reassignment_prob': 0.25,
        'diffusion_probability': 0.25,
        'cell_compactness': 0.3,
        'num_threads': 64,
    },
    {
        'label': 'compact_1.0',
        'samples': 500,
        'voxel_size': 2.0,
        'burnin_voxel_size': 4.0,
        'voxel_layers': voxel_layers_default,
        'nuclear_reassignment_prob': 0.25,
        'diffusion_probability': 0.25,
        'cell_compactness': 1.0,
        'num_threads': 64,
    },
]

# Expansion-focused sweep: vary parameters that actually grow seed masks.
base_proseg = {
    'samples': 1200,
    'voxel_size': 0.5,
    'burnin_voxel_size': 1.0,
    'voxel_layers': voxel_layers_default,
    'nuclear_reassignment_prob': 0.25,
    'diffusion_probability': 0.25,
    'cell_compactness': 0.04,
    'expand_initialized_cells': 0,
    'prior_seg_reassignment_prob': 0.5,
    'use_cell_initialization': True,
    'max_transcript_nucleus_distance': 60.0,
    'diffusion_sigma_far': None,
    'num_threads': 64,
}

proseg_param_sets = [
    {**base_proseg, 'label': 'base'},
    {**base_proseg, 'label': 'expand_1', 'expand_initialized_cells': 1},
    {**base_proseg, 'label': 'expand_2', 'expand_initialized_cells': 2},
    {
        **base_proseg,
        'label': 'expand_2_reassign_0p8',
        'expand_initialized_cells': 2,
        'prior_seg_reassignment_prob': 0.8,
    },
    {
        **base_proseg,
        'label': 'expand_2_diffsig_6',
        'expand_initialized_cells': 2,
        'diffusion_sigma_far': 6.0,
    },
    {
        **base_proseg,
        'label': 'expand_2_maxdist_90',
        'expand_initialized_cells': 2,
        'max_transcript_nucleus_distance': 90.0,
    },
    {
        **base_proseg,
        'label': 'expand_2_maxdist_120',
        'expand_initialized_cells': 2,
        'max_transcript_nucleus_distance': 120.0,
    },
    {
        **base_proseg,
        'label': 'expand_3_maxdist_120_reassign_0p9',
        'expand_initialized_cells': 3,
        'max_transcript_nucleus_distance': 120.0,
        'prior_seg_reassignment_prob': 0.9,
    },
]

if run_proseg:
    proseg_output_dir.mkdir(parents=True, exist_ok=True)

    trans = pts_gdf[['x', 'y', 'z', 'gene', 'cell_id']].copy()
    trans['x_micron'] = trans['x'].astype(float)
    trans['y_micron'] = trans['y'].astype(float)
    trans['feature_name'] = trans['gene'].astype(str)
    trans['cell_id'] = trans['cell_id'].fillna('0').astype(str)
    trans['z_micron'] = pd.to_numeric(trans['z'], errors='coerce').fillna(0).astype(float)
    n_seeded = int((trans['cell_id'] != '0').sum())
    print('Seeded transcripts for ProSeg:', n_seeded, f"({n_seeded / max(len(trans), 1):.2%})")

    proseg_results = []
    for params in proseg_param_sets:
        label = params['label']
        out_path = proseg_output_dir / f"proseg_{label}.zarr"

        out = run_proseg_refinement(
            transcripts_df=trans,
            output_path=out_path,
            proseg_binary=proseg_binary,
            x_col='x_micron',
            y_col='y_micron',
            z_col='z_micron',
            gene_col='feature_name',
            cell_id_col='cell_id',
            samples=params['samples'],
            burnin_voxel_size=params['burnin_voxel_size'],
            voxel_size=params['voxel_size'],
            voxel_layers=params['voxel_layers'],
            nuclear_reassignment_prob=params['nuclear_reassignment_prob'],
            diffusion_probability=params['diffusion_probability'],
            cell_compactness=params.get('cell_compactness'),
            expand_initialized_cells=params.get('expand_initialized_cells'),
            use_cell_initialization=bool(params.get('use_cell_initialization', False)),
            prior_seg_reassignment_prob=params.get('prior_seg_reassignment_prob'),
            max_transcript_nucleus_distance=params.get('max_transcript_nucleus_distance'),
            diffusion_sigma_far=params.get('diffusion_sigma_far'),
            cellpose_masks=cellpose_masks,
            cellpose_x_transform=cellpose_x_transform,
            cellpose_y_transform=cellpose_y_transform,
            num_threads=params['num_threads'],
            overwrite=True,
            logger=None,
        )

        out = Path(out)
        if write_latest_zarr:
            import shutil
            out_latest = out.with_name(f'{out.stem}_latest.zarr')
            if out_latest.exists():
                shutil.rmtree(out_latest)
            try:
                sd.read_zarr(out).write(out_latest)
                out_for_plot = out_latest
                print(f'ProSeg output [{label}] (raw):', out)
                print(f'ProSeg output [{label}] (latest format):', out_for_plot)
            except Exception as e:
                out_for_plot = out
                print(f'ProSeg output [{label}] (raw):', out)
                print('Could not rewrite to latest SpatialData format; using raw output for plotting.')
                print('Rewrite error:', repr(e))
        else:
            out_for_plot = out
            print(f'ProSeg output [{label}]:', out_for_plot)

        proseg_results.append({
            'label': label,
            'params': params,
            'path': out,
            'plot_path': out_for_plot,
        })
else:
    print('run_proseg=False (set True to execute).')

In [ ]:
# ---- Cellpose tuning cell ----
cellpose_params = {
    'model_type': 'cyto3',
    'gpu': True,
    'diameter': None,
    'flow_threshold': 0.8,
    'cellprob_threshold': -5.0,
    'tile_overlap': 0.15,
    'bsize': 256,
    'factor_rescale': 1.0,  # set >1 to downscale before segmentation
}


# Preprocess crop into 3-channel uint8 image for Cellpose.
img = crop.astype(np.float32)
p2, p98 = np.percentile(img, (2, 98))
img = np.clip((img - p2) / (p98 - p2 + 1e-8), 0, 1)
img8 = (img * 255).astype(np.uint8)
if img8.ndim == 2:
    img8 = np.stack([img8] * 3, axis=-1)
elif img8.shape[-1] == 1:
    img8 = np.repeat(img8, 3, axis=-1)
elif img8.shape[-1] == 2:
    img8 = np.concatenate([img8, np.zeros_like(img8[..., :1])], axis=-1)
elif img8.shape[-1] > 3:
    img8 = img8[..., :3]

scale_factor = float(cellpose_params['factor_rescale'])
if scale_factor > 1.0:
    target_shape = (int(img8.shape[0] / scale_factor), int(img8.shape[1] / scale_factor), img8.shape[2])
    img_seg = resize(img8, target_shape, preserve_range=True, anti_aliasing=True).astype(np.uint8)
else:
    img_seg = img8
    

In [ ]:
# ---- Compare masks across ProSeg parameter sweep ----
# Requires that the ProSeg sweep cell has run successfully.
if 'proseg_results' not in globals() or len(proseg_results) == 0:
    raise ValueError('No ProSeg sweep results found. Run the ProSeg sweep cell first.')


def _micron_to_local_px(geom, M_transform, x0_off, y0_off):
    """Transform a shapely geometry from global microns to local ROI pixels."""
    from shapely.ops import transform as shp_transform
    def _tx(x, y):
        arr = np.vstack([x, y, np.ones_like(x)])
        out = M_transform @ arr
        return out[0] - x0_off, out[1] - y0_off
    return shp_transform(_tx, geom)


def _load_refined_plot(path, M_transform=None, x0_off=0, y0_off=0):
    refined_sdata = sd.read_zarr(path)
    if len(refined_sdata.shapes) == 0:
        raise ValueError(f'No shapes found in ProSeg output: {path}')

    refined_shape_key = list(refined_sdata.shapes.keys())[0]
    refined_shapes = refined_sdata.shapes[refined_shape_key]

    try:
        refined_plot = refined_shapes[['geometry']].copy()
    except Exception:
        refined_plot = gpd.GeoDataFrame({'geometry': refined_shapes.geometry})

    if 'geometry' in refined_plot.columns:
        refined_plot = refined_plot[refined_plot.geometry.notna()].copy()
        refined_plot = refined_plot[~refined_plot.geometry.is_empty].copy()

    if M_transform is not None:
        refined_plot = refined_plot.copy()
        refined_plot['geometry'] = refined_plot['geometry'].apply(
            lambda g: _micron_to_local_px(g, M_transform, x0_off, y0_off)
        )

    return refined_plot, refined_shape_key


def _draw_boundaries(ax, gdf_like, color='cyan', linewidth=0.5):
    geoms = gdf_like.geometry if hasattr(gdf_like, 'geometry') else gdf_like
    for geom in geoms:
        if geom is None or geom.is_empty:
            continue
        if geom.geom_type == 'Polygon':
            x, y = geom.exterior.xy
            ax.plot(x, y, color=color, linewidth=linewidth)
        elif geom.geom_type == 'MultiPolygon':
            for part in geom.geoms:
                x, y = part.exterior.xy
                ax.plot(x, y, color=color, linewidth=linewidth)


# Zoom window: centered crop of the ROI image (edit these if needed).
h, w = img8.shape[:2]
zoom_w = min(2200, w)
zoom_h = min(2200, h)
cx, cy = w // 2, h // 2
x_min = max(0, cx - zoom_w // 2)
x_max = min(w, cx + zoom_w // 2)
y_min = max(0, cy - zoom_h // 2)
y_max = min(h, cy + zoom_h // 2)

plot_transcripts = True
pts_zoom = None
if plot_transcripts and 'pts_gdf' in globals() and {'x_local_px', 'y_local_px'}.issubset(set(pts_gdf.columns)):
    pts_zoom = pts_gdf[
        (pts_gdf['x_local_px'] >= x_min) & (pts_gdf['x_local_px'] <= x_max) &
        (pts_gdf['y_local_px'] >= y_min) & (pts_gdf['y_local_px'] <= y_max)
    ]

import math

n_runs = len(proseg_results)
ncols = min(3, n_runs)
nrows = math.ceil(n_runs / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, res in zip(axes, proseg_results):
    refined_plot, refined_shape_key = _load_refined_plot(res['plot_path'], M_transform=M, x0_off=x0, y0_off=y0)
    p = res.get('params', {})
    ax.imshow(img8[..., 0], cmap='gray')

    if len(cells_local) > 0:
        _draw_boundaries(ax, cells_local, color='red', linewidth=0.5)
    if len(refined_plot) > 0:
        _draw_boundaries(ax, refined_plot, color='cyan', linewidth=0.5)

    if pts_zoom is not None and len(pts_zoom) > 0:
        ax.scatter(
            pts_zoom['x_local_px'],
            pts_zoom['y_local_px'],
            s=2,
            c='lime',
            alpha=0.35,
            linewidths=0,
            rasterized=True,
        )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_max, y_min)  # invert y for image coordinates
    title = (
        f"{res['label']} | {refined_shape_key} | n={len(refined_plot)}\n"
        f"expand={p.get('expand_initialized_cells')} | prior_reassign={p.get('prior_seg_reassignment_prob')} | "
        f"max_dist={p.get('max_transcript_nucleus_distance')}\n"
        f"compact={p.get('cell_compactness')} | diff_prob={p.get('diffusion_probability')} | "
        f"diff_sigma_far={p.get('diffusion_sigma_far')} | voxel={p.get('voxel_size')}"
    )
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.axis('off')

for ax in axes[n_runs:]:
    ax.axis('off')

plt.show()

print('ProSeg runs:', n_runs)
for res in proseg_results:
    print(f"- {res['label']}: {res['plot_path']}")

## Tuning loop

Typical iterative workflow:

1. Edit ROI and rerun from ROI selection onward.
2. Edit `cellpose_params` and rerun Cellpose + downstream cells.
3. Edit mask filtering settings (`max_eccentricity`) and compare assignment rates.
4. Once satisfied on multiple ROIs, transfer those parameters into `src/params.yaml` for full-run workflow.